## Inspect the Listening Log Archive

Inspect archive contents and available disk space before extraction.
The original listening logs will support behavioral features and
daily sequences for later model comparisons.

In [1]:
from pathlib import Path
import shutil
import py7zr

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
archive_path = RAW_DIR / "user_logs.csv.7z"

assert archive_path.exists(), "Listening log archive not found."

gib = 1024 ** 3

print(f"Archive size: {archive_path.stat().st_size / gib:.2f} GiB")

with py7zr.SevenZipFile(archive_path, mode="r") as archive:
    archive_files = archive.list()

total_uncompressed_bytes = 0

for entry in archive_files:
    size = entry.uncompressed or 0
    total_uncompressed_bytes += size
    print(f"{entry.filename}: {size / gib:.2f} GiB")

free_bytes = shutil.disk_usage(RAW_DIR).free

print(
    f"Total uncompressed size: "
    f"{total_uncompressed_bytes / gib:.2f} GiB"
)
print(f"Available disk space: {free_bytes / gib:.2f} GiB")

Archive size: 6.65 GiB
user_logs.csv: 28.42 GiB
Total uncompressed size: 28.42 GiB
Available disk space: 40.87 GiB


In [2]:
log_path = RAW_DIR / "user_logs.csv"
reserve_bytes = 4 * gib

if log_path.exists():
    assert log_path.stat().st_size == total_uncompressed_bytes, (
        "An existing log file has an unexpected size. "
        "Check whether a previous extraction was interrupted."
    )
    print("The extracted log file already exists.")
else:
    available_bytes = shutil.disk_usage(RAW_DIR).free

    assert available_bytes >= (
        total_uncompressed_bytes + reserve_bytes
    ), "Insufficient disk space for extraction and working files."

    print("Extracting user_logs.csv...", flush=True)

    with py7zr.SevenZipFile(archive_path, mode="r") as archive:
        archive.extract(
            path=RAW_DIR,
            targets=["user_logs.csv"],
        )

    assert log_path.stat().st_size == total_uncompressed_bytes, (
        "The extracted file size does not match the archive metadata."
    )

    print("Extraction complete.")

print(f"CSV size: {log_path.stat().st_size / gib:.2f} GiB")
print(
    f"Remaining disk space: "
    f"{shutil.disk_usage(RAW_DIR).free / gib:.2f} GiB"
)

Extracting user_logs.csv...
Extraction complete.
CSV size: 28.42 GiB
Remaining disk space: 12.47 GiB


In [3]:
import pandas as pd

logs_preview = pd.read_csv(log_path, nrows=5)

print("Columns:", logs_preview.columns.tolist())
display(logs_preview)
display(logs_preview.dtypes.rename("preview_dtype").to_frame())

Columns: ['msno', 'date', 'num_25', 'num_50', 'num_75', 'num_985', 'num_100', 'num_unq', 'total_secs']


,msno,date,num_25,num_50,num_75,num_985,num_100,num_unq,total_secs
0,rxIP2f2aN0rYNp+toI0Obt/N/FYQX8hcO1fTmmy2h34=,20150513,0,0,0,0,1,1,280.335
1,rxIP2f2aN0rYNp+toI0Obt/N/FYQX8hcO1fTmmy2h34=,20150709,9,1,0,0,7,11,1658.948
2,yxiEWwE9VR5utpUecLxVdQ5B7NysUPfrNtGINaM2zA8=,20150105,3,3,0,0,68,36,17364.956
3,yxiEWwE9VR5utpUecLxVdQ5B7NysUPfrNtGINaM2zA8=,20150306,1,0,1,1,97,27,24667.317
4,yxiEWwE9VR5utpUecLxVdQ5B7NysUPfrNtGINaM2zA8=,20150501,3,0,0,0,38,38,9649.029


,preview_dtype
msno,str
date,int64
num_25,int64
num_50,int64
num_75,int64
num_985,int64
num_100,int64
num_unq,int64
total_secs,float64


## Audit Dates and Listening Values

Scan the full CSV for date coverage, missing identifiers,
unparseable or negative listening counts, and unusual durations.
Flagged values are retained for investigation.

In [4]:
import duckdb

temp_dir = PROJECT_ROOT / "data" / "duckdb_tmp"
temp_dir.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(config={
    "threads": 2,
    "memory_limit": "1GB",
    "temp_directory": str(temp_dir),
    "max_temp_directory_size": "2GB",
})

print("DuckDB audit connection ready.")

DuckDB audit connection ready.


In [5]:
count_columns = [
    "num_25",
    "num_50",
    "num_75",
    "num_985",
    "num_100",
    "num_unq",
]

count_casts = ",\n".join(
    f"TRY_CAST({column} AS BIGINT) AS {column}"
    for column in count_columns
)

invalid_count_condition = " OR ".join(
    f"{column} IS NULL"
    for column in count_columns
)

negative_count_condition = " OR ".join(
    f"{column} < 0"
    for column in count_columns
)

logs_audit_sql = f"""
WITH parsed AS (
    SELECT
        msno,
        TRY_STRPTIME("date", '%Y%m%d')::DATE AS log_day,
        {count_casts},
        TRY_CAST(total_secs AS DOUBLE) AS listening_seconds
    FROM read_csv(?, all_varchar = true)
)
SELECT
    COUNT(*) AS rows,
    MIN(log_day) AS first_log_date,
    MAX(log_day) AS last_log_date,
    COUNT(*) FILTER (
        WHERE msno IS NULL OR TRIM(msno) = ''
    ) AS missing_user_ids,
    COUNT(*) FILTER (
        WHERE log_day IS NULL
    ) AS missing_or_invalid_dates,
    COUNT(*) FILTER (
        WHERE {invalid_count_condition}
    ) AS rows_with_missing_or_invalid_counts,
    COUNT(*) FILTER (
        WHERE {negative_count_condition}
    ) AS rows_with_negative_counts,
    COUNT(*) FILTER (
        WHERE listening_seconds IS NULL
           OR NOT ISFINITE(listening_seconds)
    ) AS rows_with_missing_or_invalid_seconds,
    COUNT(*) FILTER (
        WHERE ISFINITE(listening_seconds)
          AND listening_seconds < 0
    ) AS rows_with_negative_seconds,
    COUNT(*) FILTER (
        WHERE listening_seconds = 0
    ) AS rows_with_zero_seconds,
    COUNT(*) FILTER (
        WHERE ISFINITE(listening_seconds)
          AND listening_seconds > 86400
    ) AS rows_above_24_hours,
    MIN(listening_seconds) FILTER (
        WHERE ISFINITE(listening_seconds)
    ) AS min_finite_seconds,
    MAX(listening_seconds) FILTER (
        WHERE ISFINITE(listening_seconds)
    ) AS max_finite_seconds
FROM parsed
"""

print("Scanning the full listening log file...", flush=True)

logs_audit = con.execute(
    logs_audit_sql,
    [str(log_path)],
).df()

display(logs_audit.T)

Scanning the full listening log file...


,0
rows,392106543
first_log_date,2015-01-01 00:00:00
last_log_date,2017-02-28 00:00:00
missing_user_ids,0
missing_or_invalid_dates,0
rows_with_missing_or_invalid_counts,0
rows_with_negative_counts,0
rows_with_missing_or_invalid_seconds,0
rows_with_negative_seconds,61493
rows_with_zero_seconds,0


In [6]:
import json

report_dir = PROJECT_ROOT / "reports"
sql_dir = PROJECT_ROOT / "sql"

report_dir.mkdir(parents=True, exist_ok=True)
sql_dir.mkdir(parents=True, exist_ok=True)

logs_audit.to_csv(
    report_dir / "user_logs_audit.csv",
    index=False,
)

(sql_dir / "user_logs_audit.sql").write_text(
    logs_audit_sql.strip() + "\n",
    encoding="utf-8",
)

logs_quality_policy = {
    "source": "user_logs.csv",
    "negative_seconds": "set duration to missing; retain song counts",
    "seconds_above_86400": (
        "provisionally set duration to missing; retain song counts"
    ),
    "duration_upper_bound": 86400,
    "preserve_duration_anomaly_indicator": True,
    "threshold_sensitivity_checked": False,
    "user_day_duplicates_checked": False,
}

(report_dir / "user_logs_quality_policy.json").write_text(
    json.dumps(logs_quality_policy, indent=2) + "\n",
    encoding="utf-8",
)

print("Listening log audit and provisional quality policy saved.")

Listening log audit and provisional quality policy saved.


## Listening Log Audit

Audited 392,106,543 listening records covering January 1, 2015 through February 28, 2017.

No missing user IDs, invalid dates, missing or unparseable song counts, negative song counts, or non-finite listening durations were found.

Listening duration anomalies include:

- 61,493 rows with negative durations.
- 142,993 rows with durations exceeding 86,400 seconds.
- Extreme values near ±9.22 × 10¹⁸ seconds.

The provisional processing policy retains song counts and represents negative or above-24-hour durations as missing, with an anomaly indicator. The upper bound is an analytical assumption requiring sensitivity checks, not a confirmed data-validity rule.

Duplicate user-day checks will be performed on the extracted modeling windows.

The full CSV is scanned with DuckDB rather than loaded into pandas. Raw logs remain local and are excluded from Git.

See [the audit notebook](notebooks/09_user_logs_audit.ipynb), [audit results](reports/user_logs_audit.csv), and [quality policy](reports/user_logs_quality_policy.json).